In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [7]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("smart_city_citizen_activity DAV.csv")

required = [
    "Citizen_ID", "Age", "Gender", "Mode_of_Transport",
    "Home_Energy_Consumption_kWh", "Carbon_Footprint_kgCO2",
    "Charging_Station_Usage", "Steps_Walked"
]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

Saving smart_city_citizen_activity DAV.csv to smart_city_citizen_activity DAV.csv


In [8]:
numeric_cols = [
    "Age", "Home_Energy_Consumption_kWh", "Carbon_Footprint_kgCO2",
    "Charging_Station_Usage", "Steps_Walked"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[
    "Age", "Home_Energy_Consumption_kWh", "Carbon_Footprint_kgCO2"
]).copy()

df["Age_Group"] = pd.cut(
    df["Age"], bins=[0, 18, 30, 45, 60, 120],
    labels=["0–18", "19–30", "31–45", "46–60", "61+"],
    include_lowest=True
)
df["Charging_Station_Status"] = df["Charging_Station_Usage"].apply(
    lambda value: "Used charging station" if value > 0 else "No recorded usage"
)


In [9]:
print("\\n===== SUSTAINABLE ENERGY CONSUMPTION DASHBOARD =====")
print(f"Citizens analysed: {len(df):,}")
print(f"Average home energy consumption: {df['Home_Energy_Consumption_kWh'].mean():.2f} kWh")
print(f"Total recorded home energy consumption: {df['Home_Energy_Consumption_kWh'].sum():.2f} kWh")
print(f"Average carbon footprint: {df['Carbon_Footprint_kgCO2'].mean():.2f} kg CO2")
print(f"Average daily steps: {df['Steps_Walked'].mean():,.0f}")

\n===== SUSTAINABLE ENERGY CONSUMPTION DASHBOARD =====
Citizens analysed: 1,000
Average home energy consumption: 6.00 kWh
Total recorded home energy consumption: 5996.90 kWh
Average carbon footprint: 54.42 kg CO2
Average daily steps: 10,515


In [10]:
fig1 = px.histogram(
    df, x="Home_Energy_Consumption_kWh", nbins=30,
    title="1. Distribution of Home Energy Consumption",
    labels={"Home_Energy_Consumption_kWh": "Home energy consumption (kWh)"},
    marginal="box", hover_data=["Citizen_ID", "Age", "Mode_of_Transport"]
)
fig1.update_layout(
    template="plotly_white",
    xaxis_title="Home energy consumption (kWh)",
    yaxis_title="Number of citizens",
    bargap=0.08
)
fig1.show()


In [11]:
fig2 = px.scatter(
    df, x="Home_Energy_Consumption_kWh", y="Carbon_Footprint_kgCO2",
    color="Mode_of_Transport", size="Steps_Walked",
    hover_data=["Citizen_ID", "Age", "Gender", "Charging_Station_Usage"],
    title="2. Home Energy Consumption vs Carbon Footprint",
    labels={
        "Home_Energy_Consumption_kWh": "Home energy consumption (kWh)",
        "Carbon_Footprint_kgCO2": "Carbon footprint (kg CO₂)",
        "Mode_of_Transport": "Transport mode",
        "Steps_Walked": "Steps walked"
    }
)
fig2.update_layout(template="plotly_white")
fig2.show()

In [12]:
fig3 = px.box(
    df, x="Mode_of_Transport", y="Home_Energy_Consumption_kWh",
    color="Mode_of_Transport", points="all",
    hover_data=["Citizen_ID", "Age", "Carbon_Footprint_kgCO2"],
    title="3. Home Energy Consumption by Transport Mode",
    labels={
        "Mode_of_Transport": "Mode of transport",
        "Home_Energy_Consumption_kWh": "Home energy consumption (kWh)"
    }
)
fig3.update_layout(template="plotly_white", showlegend=False)
fig3.show()

In [13]:
fig4 = px.scatter_3d(
    df, x="Home_Energy_Consumption_kWh", y="Carbon_Footprint_kgCO2",
    z="Steps_Walked", color="Mode_of_Transport",
    hover_data=["Citizen_ID", "Age", "Gender"],
    title="4. 3D View: Energy, Carbon Footprint and Walking Activity",
    labels={
        "Home_Energy_Consumption_kWh": "Home energy (kWh)",
        "Carbon_Footprint_kgCO2": "Carbon footprint (kg CO₂)",
        "Steps_Walked": "Steps walked"
    }
)
fig4.update_layout(template="plotly_white")
fig4.show()

In [14]:
fig5 = px.scatter(
    df.dropna(subset=["Age_Group"]),
    x="Home_Energy_Consumption_kWh", y="Carbon_Footprint_kgCO2",
    animation_frame="Age_Group",
    color="Mode_of_Transport", size="Steps_Walked",
    hover_data=["Citizen_ID", "Age", "Gender"],
    range_x=[
        float(df["Home_Energy_Consumption_kWh"].min()),
        float(df["Home_Energy_Consumption_kWh"].max())
    ],
    range_y=[
        float(df["Carbon_Footprint_kgCO2"].min()),
        float(df["Carbon_Footprint_kgCO2"].max())
    ],
    title="5. Animated Comparison Across Age Groups",
    labels={
        "Home_Energy_Consumption_kWh": "Home energy consumption (kWh)",
        "Carbon_Footprint_kgCO2": "Carbon footprint (kg CO₂)",
        "Age_Group": "Age group"
    }
)
fig5.update_layout(template="plotly_white")
fig5.show()

In [15]:
metrics = [
    ("Carbon_Footprint_kgCO2", "Carbon footprint (kg CO₂)"),
    ("Steps_Walked", "Steps walked"),
    ("Age", "Age (years)")
]
fig6 = go.Figure()
for i, (metric, label) in enumerate(metrics):
    fig6.add_trace(go.Scatter(
        x=df["Home_Energy_Consumption_kWh"],
        y=df[metric],
        mode="markers",
        name=label,
        visible=(i == 0),
        text=df["Mode_of_Transport"].astype(str),
        customdata=df[["Citizen_ID", "Gender"]].astype(str).values,
        hovertemplate=(
            "Energy: %{x:.2f} kWh<br>"
            + label + ": %{y}<br>"
            + "Transport: %{text}<br>"
            + "Citizen ID: %{customdata[0]}<br>"
            + "Gender: %{customdata[1]}<extra></extra>"
        )
    ))

buttons = []
for i, (metric, label) in enumerate(metrics):
    visible = [j == i for j in range(len(metrics))]
    buttons.append(dict(
        label=label,
        method="update",
        args=[
            {"visible": visible},
            {"yaxis": {"title": label}, "title": f"6. Energy Consumption vs {label}"}
        ]
    ))

fig6.update_layout(
    template="plotly_white",
    title="6. Energy Consumption vs Carbon Footprint",
    xaxis_title="Home energy consumption (kWh)",
    yaxis_title="Carbon footprint (kg CO₂)",
    updatemenus=[dict(
        buttons=buttons, direction="down", x=1.02, y=1,
        xanchor="left", yanchor="top"
    )],
    margin=dict(r=240)
)
fig6.show()

In [16]:
print("\\nDashboard charts created: histogram, scatter, box plot, 3D scatter, age-group animation, dropdown comparison.")
print("Interpretation reminder: kWh is energy consumption; kg CO2 is carbon footprint.")
print("This dataset does not identify renewable energy sources or provide a time column.")

\nDashboard charts created: histogram, scatter, box plot, 3D scatter, age-group animation, dropdown comparison.
Interpretation reminder: kWh is energy consumption; kg CO2 is carbon footprint.
This dataset does not identify renewable energy sources or provide a time column.
